# HW13 — токенизация, инференс BERT-подобной модели, fine-tuning (emotion)

Датасет: `emotion` (HuggingFace `datasets`) — классификация коротких текстов по **6** эмоциям.

Пайплайн: sanity-check данных → разбор токенизации → инференс **до** дообучения (предобученный энкодер + случайная голова) → `Trainer` + выбор по `validation` → финальная оценка на `test` (один раз) → артефакты.

**Почему `emotion`:** небольшой, 6 классов, есть официальные `train` / `validation` / `test`, без ручной разметки.

In [1]:
# Если ядро Jupyter не находит `datasets` / `transformers`, установите пакеты **в то же окружение**, где зарегистрировано ядро:
# `%pip install -q datasets transformers torch scikit-learn matplotlib pandas accelerate`

In [2]:
import random
import inspect
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# --- seed: единая точка для воспроизводимости сплитов/инициализации (где применимо) ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# `device` используется для ручного инференса; Trainer сам переносит модель на accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# Корень HW13: при Run All из `homeworks/HW13/` или из корня репозитория артефакты попадают в `homeworks/HW13/artifacts/`
def _resolve_hw13_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "HW13.ipynb").exists():
        return cwd
    p = cwd / "homeworks" / "HW13"
    if p.is_dir() and (p / "HW13.ipynb").exists():
        return p
    return cwd

BASE = _resolve_hw13_root()
ARTIFACT_DIR = BASE / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("BASE:", BASE.resolve(), "| ARTIFACT_DIR:", ARTIFACT_DIR.resolve())

ModuleNotFoundError: No module named 'datasets'

## 1. Данные и sanity-check

In [ ]:
# Загрузка из HuggingFace Hub (публичный учебный датасет, без ПДн)
raw = load_dataset("emotion")

label_names = raw["train"].features["label"].names
id2label = {i: name for i, name in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(label_names)

print("Классы:", label_names)
for split in raw:
    print(f"{split}: {len(raw[split])} примеров")

# 3–5 примеров
sample = raw["train"].shuffle(seed=SEED).select(range(5))
for row in sample:
    print("label:", id2label[row["label"]], "| text:", row["text"][:120], "...")

# Постановка: по какому признаку эмоциональная классификация короткого текста (английский твиттер-подобный стиль).

NameError: name 'load_dataset' is not defined

## 2. Токенизация (BERT-подобный токенизатор)

Используем `distilbert-base-uncased`: компромисс скорость/качество для учебного прогона.

**Почему показываем `input_ids` / `attention_mask`:** модель получает не «слова», а индексы в субсловарном словаре + маску реальных токенов; padding без маски искажал бы self-attention.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

demo_texts = raw["train"].shuffle(seed=SEED).select(range(4))["text"]

for i, text in enumerate(demo_texts):
    enc = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128,
        return_tensors="pt",
    )
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    print("=== example", i, "===")
    print("tokens (first 20):", tokens[:20])
    print("input_ids (first 20):", enc["input_ids"][0, :20].tolist())
    print("attention_mask (first 20):", enc["attention_mask"][0, :20].tolist())
    print("special tokens map:", tokenizer.special_tokens_map)
    print()

# Явный пример пары длин: padding + truncation
long = "word " * 400
enc_both = tokenizer(long, truncation=True, max_length=32, padding="max_length", return_tensors="pt")
print("trunc+pad length:", enc_both["input_ids"].shape)
print("last positions are PAD:", tokenizer.convert_ids_to_tokens(enc_both["input_ids"][0])[-5:])

## 3. Инференс готовой pretrained модели (до fine-tuning)

Загружаем `AutoModelForSequenceClassification` с **предобученным энкодером** и **новой** классификационной головой на `num_labels` классов. Веса головы инициализируются случайно, поэтому «сырой» инференс без обучения **не** должен совпадать с метками — это иллюстрирует, что одного предобучения энкодера недостаточно для конкретной задачи.

In [ ]:
model_pre = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)
model_pre.to(device)
model_pre.eval()

infer_texts = raw["train"].shuffle(seed=SEED).select(range(5))["text"]

with torch.no_grad():
    for t in infer_texts:
        enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=128)
        enc = {k: v.to(device) for k, v in enc.items()}
        logits = model_pre(**enc).logits
        probs = torch.softmax(logits, dim=-1)[0]
        pred_id = int(torch.argmax(probs))
        print("text:", t[:80], "...")
        print("pred:", id2label[pred_id], "conf:", float(probs[pred_id]))
        print()

## 4. Fine-tuning для sequence classification

Подготовка датасета: токенизация без padding на этапе `map` (его даст `DataCollatorWithPadding` в батче — динамический padding экономнее по памяти).

**Почему лучший чекпоинт по validation:** `test` не участвует в выборе весов; иначе оценка на `test` была бы оптимистично смещена.

In [ ]:
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized = raw.map(tokenize_fn, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized = tokenized.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
    }

training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "trainer_checkpoints"),
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",
    greater_is_better=True,
    seed=SEED,
    logging_steps=50,
    report_to="none",
)

model_ft = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

_trainer_kw = dict(
    model=model_ft,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    _trainer_kw["processing_class"] = tokenizer
else:
    _trainer_kw["tokenizer"] = tokenizer
trainer = Trainer(**_trainer_kw)

trainer.train()

## 5. Финальная оценка на `test` (один раз)

In [ ]:
# Один проход по test: метрики + матрица ошибок + таблица примеров
pred_out = trainer.predict(tokenized["test"])
logits_test = pred_out.predictions
y_true = pred_out.label_ids
y_pred = np.argmax(logits_test, axis=-1)

test_acc = accuracy_score(y_true, y_pred)
test_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
print("test_accuracy:", test_acc)
print("test_f1_macro:", test_f1)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, interpolation="nearest")
ax.set_xticks(range(num_labels))
ax.set_yticks(range(num_labels))
ax.set_xticklabels(label_names, rotation=45, ha="right")
ax.set_yticklabels(label_names)
ax.set_ylabel("Истина")
ax.set_xlabel("Предсказание")
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center", color="w" if cm[i, j] > cm.max() / 2 else "black")
fig.colorbar(im, ax=ax)
fig.tight_layout()
cm_path = ARTIFACT_DIR / "confusion_matrix.png"
fig.savefig(cm_path, dpi=150)
plt.show()
print("saved:", cm_path.resolve())

In [ ]:
# Вероятности для confidence (по test)
probs_test = torch.softmax(torch.tensor(logits_test, dtype=torch.float32), dim=-1).numpy()
conf = probs_test[np.arange(len(y_pred)), y_pred]

# Исходные тексты для test
test_texts = raw["test"]["text"]

rows = []
for i in range(len(y_true)):
    rows.append({
        "text": test_texts[i],
        "true_label": id2label[int(y_true[i])],
        "pred_label": id2label[int(y_pred[i])],
        "confidence": float(conf[i]),
    })

pred_df = pd.DataFrame(rows)
pred_path = ARTIFACT_DIR / "sample_predictions.csv"
pred_df.to_csv(pred_path, index=False)
print("saved:", pred_path.resolve(), "rows:", len(pred_df))

## 6. Примеры предсказаний и ошибки (5–10 строк)

In [ ]:
# Показать 10 примеров: в первую очередь ошибочные
wrong_idx = np.where(y_true != y_pred)[0]
show_idx = list(wrong_idx[:8])
if len(show_idx) < 10:
    show_idx += list(np.where(y_true == y_pred)[0][: (10 - len(show_idx))])

for i in show_idx[:10]:
    print("---")
    print("true:", id2label[int(y_true[i])], "| pred:", id2label[int(y_pred[i])], "| conf:", round(float(conf[i]), 4))
    print(test_texts[i][:200])

# Краткий комментарий: ошибки часто на смежных эмоциях (joy/love) или из-за короткого текста / неоднозначности.